# ToxiScan — EDA & Model Training Notebook

**Goal**: Predict drug toxicity using Tox21 (labeled) + ZINC250k (negative augmentation) datasets.

**Endpoints**: 12 binary classification targets from the Tox21 challenge.

---

In [ ]:
import sys
sys.path.insert(0, '../backend')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
CYAN = '#00e5ff'
RED  = '#ff3b6b'
GREEN = '#00ff9d'

print('Environment ready!')

## 1. Load Datasets

In [ ]:
from utils.dataset_loader import load_tox21, load_zinc250k, TOX21_ENDPOINTS

tox21 = load_tox21('../backend/data/tox21.csv')
zinc  = load_zinc250k('../backend/data/zinc250k.csv', sample_size=5000)

print('Tox21 shape:', tox21.shape)
print('ZINC250k shape:', zinc.shape)
tox21.head()

## 2. Label Distribution (Tox21)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, ep in enumerate(TOX21_ENDPOINTS):
    if ep not in tox21.columns:
        continue
    counts = tox21[ep].value_counts(dropna=False)
    ax = axes[i]
    colors = [RED if k == 1 else (GREEN if k == 0 else 'gray') for k in counts.index]
    ax.bar([str(k) for k in counts.index], counts.values, color=colors, alpha=0.85)
    ax.set_title(ep, fontsize=9, color='white')
    ax.set_xlabel('Label (0=safe, 1=toxic, NaN)', fontsize=7)
    ax.tick_params(labelsize=7)
    
    # Add toxic % annotation
    total = counts.get(0, 0) + counts.get(1, 0)
    toxic = counts.get(1, 0)
    if total > 0:
        ax.set_title(f'{ep}\n{toxic/total*100:.1f}% toxic', fontsize=8, color=CYAN)

plt.tight_layout()
plt.suptitle('Tox21 — Label Distribution per Endpoint', y=1.01, fontsize=14, color='white')
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight', facecolor='#04060d')
plt.show()

## 3. Molecular Descriptor EDA

In [ ]:
from utils.molecular import compute_descriptors_batch

print('Computing descriptors for a sample of 500 Tox21 molecules...')
sample_df = tox21.sample(min(500, len(tox21)), random_state=42)
desc_df = compute_descriptors_batch(sample_df['smiles'].tolist(), show_progress=True)
desc_df.head()

In [ ]:
# Distribution of key physicochemical properties
key_props = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 
             'NumRotatableBonds', 'NumAromaticRings', 'qed']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, prop in enumerate(key_props):
    if prop not in desc_df.columns:
        continue
    data = desc_df[prop].dropna()
    axes[i].hist(data, bins=40, color=CYAN, alpha=0.7, edgecolor='none')
    axes[i].axvline(data.median(), color=RED, linestyle='--', linewidth=1.5, label=f'Median: {data.median():.2f}')
    axes[i].set_title(prop, fontsize=10, color='white')
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

plt.tight_layout()
plt.suptitle('Molecular Descriptor Distributions (Tox21 Sample)', y=1.01, fontsize=13, color='white')
plt.savefig('descriptor_distributions.png', dpi=150, bbox_inches='tight', facecolor='#04060d')
plt.show()

In [ ]:
# Lipinski drug-likeness — compare Tox21 vs ZINC250k
zinc_sample = zinc.sample(min(500, len(zinc)), random_state=42)
zinc_desc = compute_descriptors_batch(zinc_sample['smiles'].tolist(), show_progress=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, df, label, color in [
    (axes[0], desc_df, 'Tox21', CYAN),
    (axes[1], zinc_desc, 'ZINC250k', '#b47aff'),
]:
    mw = df['MolWt'].dropna()
    logp = df['LogP'].dropna()
    ax.scatter(mw, logp, alpha=0.4, s=15, c=color, label=label)
    ax.axvline(500, color=RED, linestyle='--', linewidth=1, label='MW=500')
    ax.axhline(5, color=RED, linestyle='--', linewidth=1, label='LogP=5')
    ax.set_xlabel('Molecular Weight (Da)', color='white')
    ax.set_ylabel('LogP', color='white')
    ax.set_title(f'{label} — Lipinski Space', color='white')
    ax.legend(fontsize=8)
    ax.tick_params(colors='gray')

plt.tight_layout()
plt.savefig('lipinski_space.png', dpi=150, bbox_inches='tight', facecolor='#04060d')
plt.show()

## 4. Correlation Heatmap (Key Descriptors)

In [ ]:
corr_cols = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors',
             'NumRotatableBonds', 'NumAromaticRings', 'HeavyAtomCount', 'qed']
corr_data = desc_df[[c for c in corr_cols if c in desc_df.columns]].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, ax=ax,
    annot_kws={'size': 9}, linewidths=0.5,
)
ax.set_title('Molecular Descriptor Correlation Matrix', fontsize=13, color='white', pad=12)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor='#04060d')
plt.show()

## 5. Full Training Pipeline

In [ ]:
from models.train import train_pipeline

model, preprocessor, results = train_pipeline(
    tox21_path='../backend/data/tox21.csv',
    zinc_path='../backend/data/zinc250k.csv',
)

## 6. Evaluation Results

In [ ]:
# Plot AUC-ROC per endpoint
ep_results = {k: v for k, v in results.items() if isinstance(v, dict) and 'auc_roc' in v}
endpoints = list(ep_results.keys())
aucs = [ep_results[e]['auc_roc'] for e in endpoints]
f1s  = [ep_results[e]['f1']      for e in endpoints]

x = np.arange(len(endpoints))
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(x - 0.2, aucs, 0.35, label='AUC-ROC', color=CYAN, alpha=0.85)
bars2 = ax.bar(x + 0.2, f1s,  0.35, label='F1-Score', color=GREEN, alpha=0.85)
ax.axhline(0.7, color=RED, linestyle='--', linewidth=1, label='Baseline (0.7)')
ax.axhline(results['mean_auc_roc'], color=CYAN, linestyle=':', linewidth=1.5,
           label=f"Mean AUC={results['mean_auc_roc']:.3f}")
ax.set_xticks(x)
ax.set_xticklabels(endpoints, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', color='white')
ax.set_title('Model Performance per Tox21 Endpoint', fontsize=13, color='white')
ax.legend(fontsize=9)
ax.tick_params(colors='gray')
plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight', facecolor='#04060d')
plt.show()

print(f'\n✅ Mean AUC-ROC: {results["mean_auc_roc"]:.4f}')
print(f'✅ Mean F1:      {results["mean_f1"]:.4f}')

## 7. SHAP Global Feature Importance

In [ ]:
import shap
import joblib
import json

# Load artifacts
model_loaded = joblib.load('../backend/artifacts/model.joblib')
prep_loaded  = joblib.load('../backend/artifacts/preprocessor.joblib')
with open('../backend/artifacts/feature_meta.json') as f:
    meta = json.load(f)

print('Model loaded. Generating SHAP summary for NR-AhR endpoint (index 2)...')

In [ ]:
# Use a small background sample for speed
# (In production, use a larger background set)
from utils.molecular import compute_descriptors_batch

bg_smiles = tox21['smiles'].sample(200, random_state=42).tolist()
bg_desc   = compute_descriptors_batch(bg_smiles, show_progress=False)
bg_valid  = bg_desc[bg_desc['valid'] == True].drop(columns=['valid', 'smiles'])
X_bg = prep_loaded.transform(bg_valid)

# SHAP for first endpoint
explainer = shap.TreeExplainer(model_loaded.estimators_[2])  # NR-AhR
shap_vals = explainer.shap_values(X_bg)

# Summary plot
shap.summary_plot(
    shap_vals[1] if isinstance(shap_vals, list) else shap_vals,
    X_bg,
    feature_names=meta['feature_names_out'],
    max_display=20,
    show=False,
)
plt.title('SHAP Feature Importance — NR-AhR Endpoint', fontsize=12, color='white')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight', facecolor='#04060d')
plt.show()

## 8. Single Molecule Prediction Demo

Let's test the inference pipeline on a known toxic compound (Bisphenol A).

In [ ]:
from models.predict import predictor

test_cases = {
    'Aspirin (safe)':      'CC(=O)Oc1ccccc1C(=O)O',
    'Bisphenol A (toxic)': 'CC(c1ccc(O)cc1)(c1ccc(O)cc1)C',
    'Dioxin (toxic)':      'Clc1cc2c(cc1Cl)Oc1cc(Cl)c(Cl)cc1O2',
    'Caffeine (safe)':     'Cn1c(=O)c2c(ncn2C)n(c1=O)C',
}

for name, smiles in test_cases.items():
    result = predictor.predict(smiles)
    print(f"\n{'='*50}")
    print(f"Molecule : {name}")
    print(f"SMILES   : {smiles[:60]}...")
    print(f"Score    : {result['overall_toxicity_score']:.3f}  → {result['overall_risk']} Risk")
    print(f"Highest  : {result['highest_risk_endpoint']} ({result['highest_risk_probability']:.3f})")